In [ ]:
#!/usr/bin/env python3
"""
Description: Preprocessing of housing prices and energy consumption datasets.
"""
import os

print("Current working directory:", os.getcwd(), flush=True)
current_dir = os.getcwd()
parent_dir = os.path.dirname(os.path.dirname(current_dir))
os.chdir(parent_dir)

### Source Energy Consumption: https://assets.publishing.service.gov.uk/media/6762f39cff2c870561bde826/Postcode_level_all_meters_electricity_2023.csv/preview

### Source Housing Prices: https://www.kaggle.com/datasets/jakewright/house-price-data

### Energy Consumption

In [ ]:
import contextily as ctx
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np

# read csv file
import pandas as pd
import torch
from shapely.geometry import MultiPoint, Point

electricity_df = pd.read_csv(
    "svi_data/place-pulse-2.0/downstreamtask_data/London_Energy_Usage/Postcode_level_all_meters_electricity_2023.csv"
)
zip_df = pd.read_csv(
    "svi_data/place-pulse-2.0/downstreamtask_data/London_Energy_Usage/ukpostcodes.csv"
)
electricity_gdf = pd.merge(
    electricity_df, zip_df, how="inner", left_on="Postcode", right_on="postcode"
)
geometry = [Point(xy) for xy in zip(electricity_gdf["longitude"], electricity_gdf["latitude"])]
electricity_gdf = gpd.GeoDataFrame(electricity_gdf, geometry=geometry, crs="EPSG:4326")
loations_df = pd.read_csv("svi_data/place-pulse-2.0/h5_index.csv")
split_columns = loations_df["gsv_img"].str.split("_", expand=True)
loations_df["lat"] = split_columns[0].astype(float)
loations_df["lon"] = split_columns[1].astype(float)
loations_df["city"] = split_columns[3]
gdf_all = gpd.GeoDataFrame(
    loations_df,
    crs="EPSG:4326",
    geometry=gpd.points_from_xy(loations_df["lon"], loations_df["lat"]),
)
gdf_london = gdf_all[gdf_all["city"].str.lower() == "london"]
gdf_london = gdf_london.to_crs(epsg=3857)
fig, ax = plt.subplots(figsize=(8, 8))
gdf_london.plot(ax=ax, color="red", markersize=100)
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
ax.set_title("London Location on Basemap")
ax.axis("off")
plt.show()
multipoint = MultiPoint(gdf_all[gdf_all["city"].str.lower() == "london"].geometry.tolist())
convex_hull = multipoint.convex_hull
gdf_hull = gpd.GeoDataFrame(index=[0], geometry=[convex_hull], crs="EPSG:4326").to_crs(epsg=3857)
fig, ax = plt.subplots(figsize=(8, 8))
gdf_all[gdf_all["city"].str.lower() == "london"].to_crs(epsg=3857).plot(
    ax=ax, color="red", markersize=20
)
gdf_hull.plot(ax=ax, color="blue", alpha=0.3, edgecolor="black")
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
ax.set_title("Convex Hull of London Points")
ax.axis("off")
plt.show()
convex_hull_gdf = gpd.GeoDataFrame(geometry=[convex_hull], crs="EPSG:4326")
electricity_gdf = electricity_gdf.to_crs(epsg=4326)
filtered_electricity = electricity_gdf[electricity_gdf.within(convex_hull)]
filtered_electricity["Mean_cons_kwh_log"] = (
    filtered_electricity["Mean_cons_kwh"]
    .apply(lambda x: x if x > 0 else None)
    .apply(lambda x: np.log(x) if x is not None else None)
)
filtered_electricity["Median_cons_kwh_log"] = (
    filtered_electricity["Median_cons_kwh"]
    .apply(lambda x: x if x > 0 else None)
    .apply(lambda x: np.log(x) if x is not None else None)
)
filtered_electricity.to_csv(
    "svi_data/place-pulse-2.0/downstreamtask_data/London_Energy_Usage/London_Energy_Usage_locations.csv",
    index=False,
)

In [ ]:
gdf_plot = filtered_electricity.to_crs(epsg=3857)
fig, ax = plt.subplots(figsize=(10, 10))
gdf_plot.plot(
    ax=ax,
    column="Mean_cons_kwh_log",
    cmap="viridis",
    legend=True,
    alpha=0.7,
    edgecolor="black",
    markersize=40,
)
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
ax.set_title("Mean Electricity Consumption per Postcode in London")
ax.axis("off")
plt.show()

In [ ]:
import numpy as np

filtered_electricity["Mean_cons_kwh_log"].hist(bins=50, edgecolor="black")

### Housing prices

In [ ]:
import contextily as ctx
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np

# read csv file
import pandas as pd
import torch
from shapely.geometry import MultiPoint, Point

loations_df = pd.read_csv("svi_data/place-pulse-2.0/h5_index.csv")
split_columns = loations_df["gsv_img"].str.split("_", expand=True)
loations_df["lat"] = split_columns[0].astype(float)
loations_df["lon"] = split_columns[1].astype(float)
loations_df["city"] = split_columns[3]
gdf_all = gpd.GeoDataFrame(
    loations_df,
    crs="EPSG:4326",
    geometry=gpd.points_from_xy(loations_df["lon"], loations_df["lat"]),
)
gdf_london = gdf_all[gdf_all["city"].str.lower() == "london"]
gdf_london = gdf_london.to_crs(epsg=3857)
multipoint = MultiPoint(gdf_all[gdf_all["city"].str.lower() == "london"].geometry.tolist())
convex_hull = multipoint.convex_hull


housing_gdf = pd.read_csv(
    "svi_data/place-pulse-2.0/downstreamtask_data/London_Housing_Prices/kaggle_london_house_price_data.csv"
)
geometry = [Point(xy) for xy in zip(housing_gdf["longitude"], housing_gdf["latitude"])]
housing_gdf = gpd.GeoDataFrame(housing_gdf, geometry=geometry, crs="EPSG:4326")
housing_gdf = housing_gdf[housing_gdf.within(convex_hull)]
housing_gdf["history_date"] = pd.to_datetime(housing_gdf["history_date"])
housing_gdf = housing_gdf[housing_gdf["history_date"].dt.year == 2023]
housing_gdf["history_price_log"] = (
    housing_gdf["history_price"]
    .apply(lambda x: x if x > 0 else None)
    .apply(lambda x: np.log(x) if x is not None else None)
)
gdf_plot = housing_gdf.to_crs(epsg=3857)
fig, ax = plt.subplots(figsize=(10, 10))
gdf_plot.plot(
    ax=ax,
    column="history_price_log",
    cmap="viridis",
    legend=True,
    alpha=0.7,
    edgecolor="black",
    markersize=40,
)
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
ax.set_title("Housing prices in London")
ax.axis("off")
plt.show()

housing_gdf.to_csv(
    "svi_data/place-pulse-2.0/downstreamtask_data/London_Housing_Prices/London_Housing_Prices_locations.csv",
    index=False,
)

In [ ]:
housing_gdf["history_price_log"].hist(bins=50, edgecolor="black")

In [ ]:
housing_gdf = pd.read_csv(
    "svi_data/place-pulse-2.0/downstreamtask_data/London_Housing_Prices/kaggle_london_house_price_data.csv"
)
geometry = [Point(xy) for xy in zip(housing_gdf["longitude"], housing_gdf["latitude"])]
housing_gdf = gpd.GeoDataFrame(housing_gdf, geometry=geometry, crs="EPSG:4326")
housing_gdf = housing_gdf[housing_gdf.within(convex_hull)]
housing_gdf["history_date"] = pd.to_datetime(housing_gdf["history_date"])
housing_gdf = housing_gdf[housing_gdf["history_date"].dt.year == 2023]
housing_gdf["history_price_log"] = (
    housing_gdf["history_price"]
    .apply(lambda x: x if x > 0 else None)
    .apply(lambda x: np.log(x) if x is not None else None)
)
gdf_plot = housing_gdf.to_crs(epsg=3857)
fig, ax = plt.subplots(figsize=(10, 10))
gdf_plot.plot(
    ax=ax,
    column="history_price_log",
    cmap="viridis",
    legend=True,
    alpha=0.7,
    edgecolor="black",
    markersize=40,
)
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
ax.set_title("Mean Electricity Consumption per Postcode in London")
ax.axis("off")
plt.show()

In [ ]:
housing_gdf["history_price"].hist(bins=50, edgecolor="black")